In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
data = pd.read_csv('/content/fraudTest.csv')

In [ ]:
data.describe()

,Unnamed: 0,cc_num,amt,zip,lat,long,city_pop,unix_time,merch_lat,merch_long,is_fraud
count,555719.000000,5.557190e+05,555719.000000,555719.000000,555719.000000,555719.000000,5.557190e+05,5.557190e+05,555719.000000,555719.000000,555719.000000
mean,277859.000000,4.178387e+17,69.392810,48842.628015,38.543253,-90.231325,8.822189e+04,1.380679e+09,38.542798,-90.231380,0.003860
std,160422.401459,1.309837e+18,156.745941,26855.283328,5.061336,13.721780,3.003909e+05,5.201104e+06,5.095829,13.733071,0.062008
min,0.000000,6.041621e+10,1.000000,1257.000000,20.027100,-165.672300,2.300000e+01,1.371817e+09,19.027422,-166.671575,0.000000
25%,138929.500000,1.800429e+14,9.630000,26292.000000,34.668900,-96.798000,7.410000e+02,1.376029e+09,34.755302,-96.905129,0.000000
50%,277859.000000,3.521417e+15,47.290000,48174.000000,39.371600,-87.476900,2.408000e+03,1.380762e+09,39.376593,-87.445204,0.000000
75%,416788.500000,4.635331e+15,83.010000,72011.000000,41.894800,-80.175200,1.968500e+04,1.385867e+09,41.954163,-80.264637,0.000000
max,555718.000000,4.992346e+18,22768.110000,99921.000000,65.689900,-67.950300,2.906700e+06,1.388534e+09,66.679297,-66.952026,1.000000


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 555719 entries, 0 to 555718
Data columns (total 23 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             555719 non-null  int64  
 1   trans_date_trans_time  555719 non-null  object 
 2   cc_num                 555719 non-null  int64  
 3   merchant               555719 non-null  object 
 4   category               555719 non-null  object 
 5   amt                    555719 non-null  float64
 6   first                  555719 non-null  object 
 7   last                   555719 non-null  object 
 8   gender                 555719 non-null  object 
 9   street                 555719 non-null  object 
 10  city                   555719 non-null  object 
 11  state                  555719 non-null  object 
 12  zip                    555719 non-null  int64  
 13  lat                    555719 non-null  float64
 14  long                   555719 non-nu

In [ ]:
data['trans_date_trans_time'] = pd.to_datetime(data['trans_date_trans_time'])
data['dob'] = pd.to_datetime(data['dob'])
data['age'] = data['trans_date_trans_time'].dt.year - data['dob'].dt.year
columns_to_drop = ['Unnamed: 0', 'unix_time']
data_clean = data.drop(columns=columns_to_drop)
data_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 555719 entries, 0 to 555718
Data columns (total 22 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   trans_date_trans_time  555719 non-null  datetime64[ns]
 1   cc_num                 555719 non-null  int64         
 2   merchant               555719 non-null  object        
 3   category               555719 non-null  object        
 4   amt                    555719 non-null  float64       
 5   first                  555719 non-null  object        
 6   last                   555719 non-null  object        
 7   gender                 555719 non-null  object        
 8   street                 555719 non-null  object        
 9   city                   555719 non-null  object        
 10  state                  555719 non-null  object        
 11  zip                    555719 non-null  int64         
 12  lat                    555719 non-null  floa

In [ ]:
print('Valores nulos por columna:/n')
print(data.isnull().sum())

Valores nulos por columna:/n
Unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
age                      0
dtype: int64


In [ ]:
fraudes = data_clean[data_clean['is_fraud'] == 1]
analisis_cat = fraudes.groupby('category').agg(cantidad_fraudes=('is_fraud', 'count'),dinero_perdido=('amt','sum')).sort_values(by='cantidad_fraudes',ascending=False)
analisis_cat

,cantidad_fraudes,dinero_perdido
category,,
shopping_net,506,503123.93
grocery_pos,485,151866.72
misc_net,267,214742.95
shopping_pos,213,188887.25
gas_transport,154,1848.42
misc_pos,72,13923.82
personal_care,70,1814.44
home,67,17260.30
kids_pets,65,1288.45


In [ ]:
import sqlite3

# 1. Creamos la conexión a la base de datos SQL en memoria (aquí definimos 'conn')
conn = sqlite3.connect('banco.db')

# 2. Guardamos tu DataFrame limpio en una tabla SQL llamada 'transacciones'
data_clean.to_sql('transacciones', conn, if_exists='replace', index=False)

# 3. Definimos la consulta de SQL puro para calcular los promedios
query_banco = """
SELECT
    category,
    COUNT(is_fraud) AS cantidad_fraudes,
    SUM(amt) AS dinero_perdido,
    SUM(amt) / COUNT(is_fraud) AS promedio_por_fraude
FROM transacciones
WHERE is_fraud = 1
GROUP BY category
ORDER BY dinero_perdido DESC;
"""

# 4. Ejecutamos la query usando la variable 'conn' que acabamos de definir arriba
resultado_sql = pd.read_sql_query(query_banco, conn)
resultado_sql

,category,cantidad_fraudes,dinero_perdido,promedio_por_fraude
0,shopping_net,506,503123.93,994.316067
1,misc_net,267,214742.95,804.280712
2,shopping_pos,213,188887.25,886.794601
3,grocery_pos,485,151866.72,313.127258
4,entertainment,59,30076.17,509.765593
5,home,67,17260.30,257.616418
6,misc_pos,72,13923.82,193.386389
7,food_dining,54,6607.54,122.361852
8,gas_transport,154,1848.42,12.002727
9,personal_care,70,1814.44,25.920571


In [ ]:
import sqlite3

# 1. Creamos la conexión a la base de datos SQL en memoria (aquí definimos 'conn')
conn = sqlite3.connect('banco.db')

# 2. Guardamos tu DataFrame limpio en una tabla SQL llamada 'transacciones'
data_clean.to_sql('transacciones', conn, if_exists='replace', index=False)

# 3. Definimos la consulta de SQL puro para calcular los promedios
query_banco = """
SELECT
    category,
    COUNT(is_fraud) AS cantidad_fraudes,
    SUM(amt) AS dinero_perdido,
    SUM(amt) / COUNT(is_fraud) AS promedio_por_fraude
FROM transacciones
WHERE is_fraud = 1
GROUP BY category
ORDER BY promedio_por_fraude DESC;
"""

# 4. Ejecutamos la query usando la variable 'conn' que acabamos de definir arriba
resultado_sql = pd.read_sql_query(query_banco, conn)
resultado_sql

,category,cantidad_fraudes,dinero_perdido,promedio_por_fraude
0,shopping_net,506,503123.93,994.316067
1,shopping_pos,213,188887.25,886.794601
2,misc_net,267,214742.95,804.280712
3,entertainment,59,30076.17,509.765593
4,grocery_pos,485,151866.72,313.127258
5,home,67,17260.30,257.616418
6,misc_pos,72,13923.82,193.386389
7,food_dining,54,6607.54,122.361852
8,personal_care,70,1814.44,25.920571
9,health_fitness,52,1058.32,20.352308


In [ ]:
# Guardar el DataFrame con la edad calculada y sin columnas basura
data_clean.to_csv('transacciones_banco_limpio.csv', index=False)
print("¡Archivo listo!")

¡Archivo listo!
